In [0]:
# Retrieve values from previous task (bronze_output)
bronze_output = dbutils.jobs.taskValues.get(taskKey="Bronze", key="bronze_output")

# get each variable
start_date = bronze_output.get("start_date","")
bronze_adls = bronze_output.get("bronze_adls","")
silver_adls = bronze_output.get("silver_adls","")

print(f"start_date : {start_date} , silver_adls:{silver_adls}")

In [0]:
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

In [0]:
'''
RUN THIS CELL ONLY TO RUN THIS NOTEBOOK SEPERATLY AND NOT PART OF WORKFLOW | PIPELINE

# Mount ADLS Gen2

tiers = ["bronze","silver","gold"]
adls_paths = {tier: f"abfss://{tier}@databricksstore88.dfs.core.windows.net/" for tier in tiers}

# Accessing ADLS paths
bronze_adls = adls_paths["bronze"]
silver_adls = adls_paths["silver"]
gold_adls = adls_paths["gold"]

dbutils.fs.ls(bronze_adls)
dbutils.fs.ls(silver_adls)
dbutils.fs.ls(gold_adls)

start_date = date.today() - timedelta(1)
end_date = date.today()

'''

In [0]:
# Load the JSON data into a Spark DataFrame
df = spark.read.option("multiline","true").json(f"{bronze_adls}{start_date}_earthquake_data.json")

In [0]:
df.head()

Row(geometry=Row(coordinates=[-119.029167175293, 37.6404991149902, 0.129999995231628], type='Point'), id='nc75436297', properties=Row(alert=None, cdi=None, code='75436297', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc75436297&format=geojson', dmin=0.002426, felt=None, gap=119, ids=',nc75436297,', mag=1.18, magType='md', mmi=None, net='nc', nst=10, place='4 km W of Mammoth Lakes, CA', rms=0.02, sig=21, sources=',nc,', status='automatic', time=1789516675120, title='M 1.2 - 4 km W of Mammoth Lakes, CA', tsunami=0, type='earthquake', types=',nearby-cities,origin,phase-data,scitech-link,', tz=None, updated=1789526541546, url='https://earthquake.usgs.gov/earthquakes/eventpage/nc75436297'), type='Feature')

In [0]:

# Reshape earthquake data
df = (
    df
    .select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
    )
)
 

In [0]:
# Validate data: Check for missing or null values

df = df.withColumn('latitude',when(isnull(col('latitude')),0).otherwise(col('latitude')))\
       .withColumn('longitude',when(isnull(col('longitude')),0).otherwise(col('longitude')))\
       .withColumn('time', when(isnull(col('time')), 0).otherwise(col('time')))    


In [0]:
# Convert 'time' and 'updated' to timestamp from Unix time
df = (
    df
    .withColumn('time', (col('time') / 1000).cast(TimestampType()))
    .withColumn('updated', (col('updated') / 1000).cast(TimestampType()))
)

In [0]:
df.head()

Row(id='nc75436297', longitude=-119.029167175293, latitude=37.6404991149902, elevation=0.129999995231628, title='M 1.2 - 4 km W of Mammoth Lakes, CA', place_description='4 km W of Mammoth Lakes, CA', sig=21, mag=1.18, magType='md', time=datetime.datetime(2026, 9, 15, 23, 57, 55, 120000), updated=datetime.datetime(2026, 9, 16, 2, 42, 21, 546000))

In [0]:
# save tranformed dataframe to silver container
silver_output_path = f"{silver_adls}earthquake_events_silver/"

In [0]:
# Append DataFrame to Silver container in Parquet format
df.write.mode('append').parquet(silver_output_path)

In [0]:
# output variable to pass to next job
dbutils.jobs.taskValues.set(key="silver_output", value = silver_output_path)

In [0]:
silver_output_path

'abfss://silver@databricksstore88.dfs.core.windows.net/earthquake_events_silver/'